In [1]:
import awkward as ak
import uproot
from coffea.nanoevents import NanoEventsFactory, NanoAODSchema
from coffea.nanoevents.methods import vector
import numpy as np

In [2]:
fileset = {
    'WJetsToLNu_Pt-400To600_MatchEWPDG20_TuneCP5_13TeV-amcatnloFXFX-pythia8____1_':
[
            "root://cmsxrootd.fnal.gov//store/mc/RunIISummer20UL18NanoAODv9/WJetsToLNu_2J_TuneCP5_13TeV-amcatnloFXFX-pythia8/NANOAODSIM/106X_upgrade2018_realistic_v16_L1v1-v1/2550000/0BD074B6-C3F6-B144-A7CE-30B822187BF8.root",
#             "root://cmsxrootd.fnal.gov//store/mc/RunIISummer20UL18NanoAODv9/WJetsToLNu_2J_TuneCP5_13TeV-amcatnloFXFX-pythia8/NANOAODSIM/106X_upgrade2018_realistic_v16_L1v1-v1/2550000/9AC5AECD-9063-D040-AA99-09F9308A0238.root",
#             "root://cmsxrootd.fnal.gov//store/mc/RunIISummer20UL18NanoAODv9/WJetsToLNu_2J_TuneCP5_13TeV-amcatnloFXFX-pythia8/NANOAODSIM/106X_upgrade2018_realistic_v16_L1v1-v1/2550000/E22B3E7A-1944-1744-93DE-EF53D0635E92.root"
        ],
    'EGamma____1_':
    [
        
            "root://cmsxrootd.fnal.gov//store/data/Run2018A/EGamma/NANOAOD/UL2018_MiniAODv2_NanoAODv9-v1/270000/F8F84CA2-7E90-2D45-98B0-2773BCDCEF1D.root",
#             "root://cmsxrootd.fnal.gov//store/data/Run2018A/EGamma/NANOAOD/UL2018_MiniAODv2_NanoAODv9-v1/270000/EB703433-936F-F942-9341-635CB46ADD71.root",
#             "root://cmsxrootd.fnal.gov//store/data/Run2018A/EGamma/NANOAOD/UL2018_MiniAODv2_NanoAODv9-v1/270000/95A159AA-5B87-DA42-B1C9-DCC030E5A0FE.root"
        
    ]
}


In [5]:
fname_data = fileset["EGamma____1_"][0]

In [6]:
events_data = NanoEventsFactory.from_root(fname_data, schemaclass=NanoAODSchema).events()

In [175]:
min(events_data.run)

315267

In [41]:
j = events.Jet

In [47]:
j.puId>=0

<Array [[True, True, True, ... True, True]] type='932888 * var * bool'>

In [49]:
j.jetId

<Array [[6, 6, 2, 0], [6, ... [6, 6, 6, 6, 6]] type='932888 * var * int32[parame...'>

In [176]:
max(events_data.run)

316876

In [3]:
fname = fileset["WJetsToLNu_Pt-400To600_MatchEWPDG20_TuneCP5_13TeV-amcatnloFXFX-pythia8____1_"][0]
events = NanoEventsFactory.from_root(fname, schemaclass=NanoAODSchema).events()

In [4]:
met = events.MET

In [5]:
npvs = events.PV.npvs
# [-(-0.195191*npvs -0.170948)]*len(events)

In [6]:
events.Pileup.nTrueInt

<Array [23, 22, 36, 22, 35, ... 25, 39, 45, 50] type='932888 * float32[parameter...'>

In [6]:
np.asarray(npvs)

array([19, 22, 20, ..., 34, 40, 47], dtype=int32)

In [134]:
np.ones(len(events))*100

array([100., 100., 100., ..., 100., 100., 100.])

In [59]:
max(npvs)

100

In [7]:
mask = np.asarray(npvs>100)
npvs = np.asarray(npvs)
# npvs[mask==True] = 100

In [31]:
[-(-0.188743*npvs +0.136539)]

[array([3.449578, 4.015807, 3.638321, ..., 6.280723, 7.413181, 8.734382])]

In [7]:
def METXYCorr_Met_MetPhi(events, year):


    #https://lathomas.web.cern.ch/lathomas/METStuff/XYCorrections/XYMETCorrection_withUL17andUL18.h
    if 'genWeight' in events.fields:
        isData = False
        print("hi I am MC")
    else: isData = True
#     dataset=events.metadata['dataset'] 
    dataset = 'WJetsToLNu_Pt-400To600_MatchEWPDG20_TuneCP5_13TeV-amcatnloFXFX-pythia8____1_'
    met = events.MET
    originalMet = met.pt
    originalMet_phi = met.phi

    npv = events.Pileup.nTrueInt
    mask = np.asarray(npv>100)
    npv = np.asarray(npv)
    npv[mask==True] = 100


    if not isData:
        if (year == '2016') and ("preVFP" in dataset):
            METxcorr = [-(-0.188743*npv +0.136539)]
            METycorr = [-(0.0127927*npv +0.117747)]

        if (year == '2016') and ("postVFP" in dataset):
            METxcorr = [-(-0.153497*npv +0.231751)]
            METycorr = [-(0.00731978*npv +0.243323)]

        elif year == '2017':
            METxcorr = [-(-0.300155*npv +1.90608)]
            METycorr = [-(0.300213*npv +2.02232)]
        elif year == '2018':
            METxcorr = [-(0.183518*npv +0.546754)]
            METycorr = [-(0.192263*npv +0.42121)]

    if isData:

#                 print(runNMax, runNMin)
        if "2018" in year:
            runNMax = max(events.run)
            runNMin = min(events.run)
            if (runNMin >=315252) and (runNMax <=316995):
                era = "A"

            elif (runNMin >=316998) and (runNMax <=319312):
                era = "B"

            elif (runNMin >=319313) and (runNMax <=320393):
                era = "C"

            elif (runNMin >=320394) and (runNMax <=325273):
                era = "D"
            if "A" in era:
                METxcorr = [-(0.263733*npv -1.91115)]
                METycorr = [-(0.0431304*npv -0.112043)]
            elif "B" in era:
                METxcorr = [-(0.400466*npv -3.05914)]
                METycorr = [-(0.146125*npv -0.533233)]
            elif "C" in era:
                METxcorr = [-(0.430911*npv -1.42865)]
                METycorr = [-(0.0620083*npv -1.46021)]
            elif "D" in era:
                METxcorr = [-(0.457327*npv -1.56856)]
                METycorr = [-(0.0684071*npv -0.928372)]


        if "2017" in year:

            runNMax = max(events.run)
            runNMin = min(events.run)
            if (runNMin >=297020) and (runNMax <=299329):
                era = "B"

            elif (runNMin >=299337) and (runNMax <=302029):
                era = "C"

            elif (runNMin >=302030) and (runNMax <=303434):
                era = "D"

            elif (runNMin >=303435) and (runNMax <=304826):
                era = "E"

            elif (runNMin >=304911) and (runNMax <=306462):
                era = "F"

            else: era = "X"
            if "E" in era:
                METxcorr = [-(-0.162472*npv +0.176329)]
                METycorr = [-(0.138076*npv -0.250239)]
            elif "B" in era:
                METxcorr = [-(-0.211161*npv +0.419333)]
                METycorr = [-(0.251789*npv -1.28089)]
            elif "C" in era:
                METxcorr = [-(-0.185184*npv -0.164009)]
                METycorr = [-(0.200941*npv -0.56853)]
            elif "D" in era:
                METxcorr = [-(-0.201606*npv +0.426502)]
                METycorr = [-(0.188208*npv -0.58313)]
            elif "F" in era:
                METxcorr = [-(-0.210639*npv +0.72934)]
                METycorr = [-(0.198626*npv +1.028)]
            elif "X" in era:
                METxcorr = [np.zeros(len(events))]
                METycorr = [np.zeros(len(events))]
        if "2016" in year:
            runNMax = max(events.run)
            runNMin = min(events.run)

            if (runNMin >=272007) and (runNMax <=275376):
                era = "B"
                print(era)
            elif (runNMin >=275656) and (runNMax <=276283):
                era = "C"
                print('runNMax', runNMax, 'runMin', runNMin, ' era:', era)
#                         print(era)
            elif (runNMin >=276315) and (runNMax <=276811):
                era = "D"
                print(era)
            elif (runNMin >=276831) and (runNMax <=277420) :
                era = "E"
                print(era)
            elif (runNMin >=277772 and runNMax <=278807) or (runNMin in [278770, 278806, 278807]) or (runNMax in [278770, 278806, 278807] ):
                era = "F"
#                  https://twiki.cern.ch/twiki/bin/viewauth/CMS/PdmVDatasetsUL2016
                print(era)
            elif (runNMin >=278801 and runNMax <=278808) or (runNMax in [278769, 278806, 278807]) or (runNMax in [278769,278806, 278807]):
                era = "F_late"
                print(era)
            elif (runNMin >=278820) and (runNMax <=280385):
                era = "G"
                print(era)
            elif (runNMin >=281613) and (runNMax <=284044):
                era = "H"
                print(era)

            if "B" in era:
                METxcorr = [-(-0.0214894*npv -0.188255)]
                METycorr = [-(0.0876624*npv +0.812885)]
            elif "C" in era:
                METxcorr = [-(-0.032209*npv +0.067288)]
                METycorr = [-(0.113917*npv +0.743906)]
            elif "D" in era:
                METxcorr = [-(-0.0293663*npv +0.21106)]
                METycorr = [-(0.11331*npv +0.815787)]
            elif "E" in era:
                METxcorr = [-(-0.0132046*npv +0.20073)]
                METycorr = [-(0.134809*npv +0.679068)] 
            elif "F" in era:
                METxcorr = [-(-0.0543566*npv +0.816597)]
                METycorr = [-(0.114225*npv +1.17266)] 
            elif "F_late" in era:
                METxcorr = [-(0.134616*npv -0.89965)]
                METycorr = [-(0.0397736*npv +1.0385)] 
            elif "G" in era:
                METxcorr = [-(0.121809*npv -0.584893)]
                METycorr = [-(0.0558974*npv +0.891234)]
            elif "H" in era:
                METxcorr = [-(0.0868828*npv -0.703489)]
                METycorr = [-(0.0888774*npv +0.902632)]

    CorrectedMET_x = np.multiply(originalMet,np.cos(originalMet_phi))+ (np.array(METxcorr)).flatten()
    CorrectedMET_y = np.multiply(originalMet,np.sin(originalMet_phi))+ (np.array(METycorr)).flatten()
#             print(CorrectedMET_x/CorrectedMET_y)
#   double CorrectedMET_x = uncormet *cos( uncormet_phi)+METxcorr;
#   double CorrectedMET_y = uncormet *sin( uncormet_phi)+METycorr;
    CorrectedMET = np.sqrt(np.square(CorrectedMET_x)+np.square(CorrectedMET_y))
    CorrectedMETPhi = np.zeros(len(events))

#             print(CorrectedMET)
#             print(CorrectedMETPhi)
    
    
    mask1 = (CorrectedMET_x ==0 )& (CorrectedMET_y > 0)
    CorrectedMETPhi[mask1==True] = np.pi

    mask2 = (CorrectedMET_x ==0 )& (CorrectedMET_y < 0)
    CorrectedMETPhi[mask2==True] = -np.pi

    mask3 = (CorrectedMET_x <0 )& (CorrectedMET_y > 0)
    CorrectedMETPhi[mask3==True] = (np.arctan(CorrectedMET_y/CorrectedMET_x) + np.pi )[mask3==True] 

    mask4 = (CorrectedMET_x <0 )& (CorrectedMET_y < 0)
    CorrectedMETPhi[mask4==True] = (np.arctan(CorrectedMET_y/CorrectedMET_x) - np.pi )[mask4==True]

    mask5 = (CorrectedMET_x >0 )
    CorrectedMETPhi[mask5==True] = (np.arctan(CorrectedMET_y/CorrectedMET_x))[mask5==True]   
    
    mask6 = (~mask1 & ~mask2 & ~mask3 & ~mask4 & ~mask5) 
    CorrectedMETPhi[mask6==True] = 0
    return (CorrectedMET, CorrectedMETPhi)

# XY_correctedmet_pt , XY_correctedmet_phi = METXYCorr_Met_MetPhi(events, self._year)

In [8]:
met_pt, met_phi=METXYCorr_Met_MetPhi(events, '2018')

hi I am MC


In [28]:
a=np.array([1,2,3]) 
b=np.array([0,1,2])

In [35]:
-np.pi

-3.141592653589793

In [ ]:

double CorrectedMET = sqrt(CorrectedMET_x*CorrectedMET_x+CorrectedMET_y*CorrectedMET_y);
double CorrectedMETPhi;
if(CorrectedMET_x==0 && CorrectedMET_y>0) CorrectedMETPhi = TMath::Pi();
else if(CorrectedMET_x==0 && CorrectedMET_y<0 )CorrectedMETPhi = -TMath::Pi();
else if(CorrectedMET_x >0) CorrectedMETPhi = TMath::ATan(CorrectedMET_y/CorrectedMET_x);
else if(CorrectedMET_x <0&& CorrectedMET_y>0) CorrectedMETPhi = TMath::ATan(CorrectedMET_y/CorrectedMET_x) + TMath::Pi();
else if(CorrectedMET_x <0&& CorrectedMET_y<0) CorrectedMETPhi = TMath::ATan(CorrectedMET_y/CorrectedMET_x) - TMath::Pi();
else CorrectedMETPhi =0;

In [9]:
met_phi

array([-0.1692664 , -2.93249011,  1.25217211, ...,  1.53963697,
        0.01062734,  1.42598641])

In [10]:
met.phi

<Array [-0.0957, -3.11, 1.16, ... 0.159, 1.32] type='932888 * float32[parameters...'>

In [11]:
met_pt

<Array [71.8, 25.9, 37.5, ... 55, 51.2, 66.4] type='932888 * float32'>

In [12]:
met.pt

<Array [75.9, 20.8, 46.9, ... 63.6, 60.7, 78.2] type='932888 * float32[parameter...'>

In [16]:
met["T"] = ak.zip({"pt": met.pt, "phi": met.phi}, 
                          with_name="PolarTwoVector", 
                          behavior=vector.behavior)

In [17]:
corrected_met = events.MET

In [18]:
corrected_met["T"] = ak.zip({"pt": met_pt, "phi": met_phi}, 
                          with_name="PolarTwoVector", 
                          behavior=vector.behavior)

In [19]:
met["T"]

<PolarTwoVectorArray [{pt: 75.9, ... phi: 1.32}] type='932888 * PolarTwoVector["...'>

In [20]:
corrected_met['T']

<PolarTwoVectorArray [{pt: 72.4, phi: -0.157, ... phi: 1.42}] type='932888 * Pol...'>

In [60]:
j = events.Jet

In [61]:
j["T"] = ak.zip({"pt": j.pt, "phi": j.phi}, 
                  with_name="PolarTwoVector", 
                  behavior=vector.behavior)

In [67]:
j['T']['phi']

<Array [[2.07, -1.78, -3.12, ... 1.48, 0.515]] type='932888 * var * float32[para...'>

In [68]:
met['T']['phi']

<Array [-0.0957, -3.11, 1.16, ... 0.159, 1.32] type='932888 * float32[parameters...'>

In [69]:
corrected_met['T']['phi']

<Array [-0.157, -2.93, 1.21, ... 0.0289, 1.42] type='932888 * float64'>

In [63]:
leading_j = j[:,:1]
met.delta_phi(leading_j)

<Array [[-2.17], [2.82], ... [-2.3], [-1.98]] type='932888 * var * float32'>

In [66]:
corrected_met['T'].delta_phi(leading_j['T'])

<Array [[-2.23], [2.99], ... [-2.43], [-1.88]] type='932888 * var * float64'>

In [214]:
np.sqrt(2*leading_e.pt*met["T"]["pt"]*(1-np.cos(met["T"].delta_phi(leading_e))))

<Array [[84], [], [], ... [], [], [3.68]] type='932888 * var * float32'>

In [213]:
np.sqrt(2*leading_e.pt*corrected_met["T"]["pt"]*(1-np.cos(corrected_met["T"].delta_phi(leading_e))))

<Array [[90], [], [], ... [], [], [7.17]] type='932888 * var * float64'>

In [154]:
with open("../../metadata/UL_2018/SingleMuon_Run2018B.txt") as f:
    data = f.readline()
    with open()

In [156]:
import json

In [159]:
with open("../../metadata/UL_2018.json") as f:
    data = json.load(f)

In [165]:
for k in data.keys():
    
    if "SingleMuon" in k:
        print(k)
        print(data[k])

SingleMuon____1_
{'files': ['root://cmsxrootd.fnal.gov//store/data/Run2018A/SingleMuon/NANOAOD/UL2018_MiniAODv2_NanoAODv9-v2/2550000/AA26E054-373D-5241-A1B8-66F0D9D76632.root', 'root://cmsxrootd.fnal.gov//store/data/Run2018A/SingleMuon/NANOAOD/UL2018_MiniAODv2_NanoAODv9-v2/2550000/B532BB97-C4AC-0042-BF53-4DE3B2916BA5.root', 'root://cmsxrootd.fnal.gov//store/data/Run2018A/SingleMuon/NANOAOD/UL2018_MiniAODv2_NanoAODv9-v2/2550000/AAA335FC-E4F4-1C4A-AC48-6866B1BA1DCA.root'], 'xs': -1}
SingleMuon____2_
{'files': ['root://cmsxrootd.fnal.gov//store/data/Run2018A/SingleMuon/NANOAOD/UL2018_MiniAODv2_NanoAODv9-v2/2550000/F962088A-97A7-B14C-B7C1-B8FC8024CED6.root', 'root://cmsxrootd.fnal.gov//store/data/Run2018A/SingleMuon/NANOAOD/UL2018_MiniAODv2_NanoAODv9-v2/2550000/0EC110C4-967D-AE49-A2F5-DD7C0369FCBA.root', 'root://cmsxrootd.fnal.gov//store/data/Run2018A/SingleMuon/NANOAOD/UL2018_MiniAODv2_NanoAODv9-v2/2550000/4C2086CE-3443-0841-9181-D10E788033D3.root'], 'xs': -1}
SingleMuon____3_
{'files': [

In [150]:
SingleMuon_RunA = []
SingleMuon_RunB = []
SingleMuon_RunC = []
SingleMuon_RunD = []

In [152]:
for file in data:
    if 'Run2018A' in file:
        SingleMuon_RunA.append(file)
    if 'Run2018B' in file:
        SingleMuon_RunB.append(file)
    if 'Run2018C' in file:
        SingleMuon_RunC.append(file)
    if 'Run2018D' in file:
        SingleMuon_RunD.append(file)

['root://cmsxrootd.fnal.gov//store/data/Run2018D/SingleMuon/NANOAOD/UL2018_MiniAODv2_NanoAODv9-v1/280000/2CBE8C1B-15FE-494F-A7AA-0F8BDC10D64B.root\n',
 'root://cmsxrootd.fnal.gov//store/data/Run2018D/SingleMuon/NANOAOD/UL2018_MiniAODv2_NanoAODv9-v1/280000/5A3D2DCB-CF10-4E4F-8E86-05725E2AB5A4.root\n',
 'root://cmsxrootd.fnal.gov//store/data/Run2018D/SingleMuon/NANOAOD/UL2018_MiniAODv2_NanoAODv9-v1/280000/B8B14257-1B6B-E847-A41E-2031BCB468C8.root\n',
 'root://cmsxrootd.fnal.gov//store/data/Run2018D/SingleMuon/NANOAOD/UL2018_MiniAODv2_NanoAODv9-v1/280000/7BC47B80-DF0A-3648-AD56-DD81B795F3FA.root\n',
 'root://cmsxrootd.fnal.gov//store/data/Run2018D/SingleMuon/NANOAOD/UL2018_MiniAODv2_NanoAODv9-v1/280000/7E6B6743-00BF-6C4F-889A-8118A1C35C90.root\n',
 'root://cmsxrootd.fnal.gov//store/data/Run2018D/SingleMuon/NANOAOD/UL2018_MiniAODv2_NanoAODv9-v1/280000/0A7B4EA1-E44D-E64A-B269-22A132D5E72E.root\n',
 'root://cmsxrootd.fnal.gov//store/data/Run2018D/SingleMuon/NANOAOD/UL2018_MiniAODv2_NanoAODv9

In [21]:
import ROOT

Welcome to JupyROOT 6.18/00


In [24]:
ROOT.TMath.ATan(10)

1.4711276743037347

In [25]:
np.arctan(10)

1.4711276743037347

In [27]:
1 - 1

0